In [8]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
from itertools import chain
from IPython.display import display, Markdown, HTML
import cProfile

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [15]:
# Original Analysis Code
def fcount(count, total):
    return f"{count} ({100*count/total:.2f} %)"


def summarize_index(data: pd.DataFrame):
    """Create a dataframe describing the index of the given data

    Args:
        data (pd.DataFrame): Each column has rows with information
    """

    def summarize_index_col(col):
        uniq = col.nunique()
        total = len(col)
        nas = col.isna().sum()
        valcounts = col.value_counts()
        res = pd.Series(
            {
                "Number of Distinct Values (Ignoring Missing Values)": fcount(
                    uniq, total
                ),
                "Number of Missing Values": fcount(nas, total),
            }
        )
        repeats = summarize_numericals(valcounts).drop(
            index=["Number of Not Missing Values"]
        )
        repeats = repeats.set_axis("Repeats " + repeats.index, axis=0)
        return pd.concat([res, repeats])

    data = data.index.to_frame().reset_index(drop=True)
    data[" + ".join(data.columns)] = data.groupby(list(data.columns)).ngroup()
    return data.agg(summarize_index_col)


def count(series):
    return fcount(series.count(), series.size)


def nunique(series):
    return fcount(series.nunique(), series.size)


def summarize_categoricals(data: pd.DataFrame, official_doc: pd.DataFrame = None):
    """Treat the columns in data as categoricals and summarize them

    Args:
        data (pd.DataFrame): The data

    Returns:
        pd.DataFrame: Each column has rows with information
    """

    def info(series):
        row_info = official_doc[official_doc["Shortnames in der BED-DB"] == series.name]
        if len(row_info) == 0:
            return "NA"
        return row_info.iloc[0][
            "Beschreibung"
        ]  # es wird der erste datensatz genommen... (eigentlich sollte es nur einer sein)

    def content(series):
        row_info = official_doc[official_doc["Shortnames in der BED-DB"] == series.name]
        if len(row_info) == 0:
            return "NA"
        return row_info.iloc[0][
            "Inhalt/Form"
        ]  # es wird der erste datensatz genommen... (eigentlich sollte es nur einer sein)

    def most_common(series):
        counts = series.value_counts(dropna=True)
        if len(counts) == 0:
            return "All NA"
        return f"'{counts.index[0]}' ({counts.iloc[0]})"

    def least_common(series):
        counts = series.value_counts(dropna=True)
        if len(counts) == 0:
            return "All NA"
        return f"'{counts.index[-1]}' ({counts.iloc[-1]})"

    def list_values(series):
        max_values = 5
        unique_values = series.dropna().unique()
        if len(unique_values) > max_values:
            return f"{len(unique_values)} unique values"

        unique_values = [f'"{v}"' for v in unique_values]
        return ", ".join(unique_values)

    aggs = [count, nunique, most_common, least_common, list_values]
    human_names = [
        "Number of Not Missing Values",
        "Number of Distinct Values (Ignoring Missing Values)",
        "Most Common Value",
        "Least Common Value",
        "All distinct Values",
    ]
    if type(official_doc) == pd.DataFrame:
        aggs.insert(0, info)
        human_names.insert(0, "Info")

        aggs.insert(1, content)
        human_names.insert(1, "Content")
    res = data.agg(aggs)
    res = res.set_axis(
        human_names,
        axis=0,
    )
    return res


def summarize_numericals(data, official_docs: pd.DataFrame = None):
    """Treat the columns in data as numericals and summarize them

    Args:
        data (pd.DataFrame): The data

    Returns:
        pd.DataFrame: Each column has rows with information
    """
    describe = data.describe().set_axis(
        [
            "Number of Not Missing Values",
            "Mean",
            "Std",
            "Min",
            "25%",
            "Median",
            "75%",
            "Max",
        ],
        axis=0,
    )
    if official_docs is not None:
        filtered_docs = official_docs.loc[
            official_docs["Shortnames in der BED-DB"].isin(describe.columns),
            ["Shortnames in der BED-DB", "Beschreibung", "Inhalt/Form"],
        ]

        filtered_docs = filtered_docs.rename(
            columns={"Inhalt/Form": "Content", "Beschreibung": "Info"}
        )
        filtered_docs = filtered_docs.set_index("Shortnames in der BED-DB").transpose()

        merged_df = pd.concat([filtered_docs, describe], axis=0)
        return merged_df

    return describe


def summarize_strings(data, official_doc: pd.DataFrame = None):
    """Treat the columns in data as text information and summarize them

    Args:
        data (pd.DataFrame): The data

    Returns:
        pd.DataFrame: Each column has rows with information
    """

    def token_analysis(series):
        split = series.str.split(r"[\W]")
        alltokens = pd.Series(chain.from_iterable(split))
        alltokens = summarize_categoricals(alltokens, official_doc).drop(
            index=["Number of Not Missing Values"]
        )
        alltokens = alltokens.set_axis("Token " + alltokens.index, axis=0)
        return alltokens

    def nominal_analysis(series, multi=False):
        row_count = series.count()
        sum_of_space = series.str.contains(" ", regex=False).sum()
        sum_of_coma = series.str.contains(",", regex=False).sum()
        sum_of_semicolon = series.str.contains(";", regex=False).sum()
        sum_of_delimiter = sum_of_space + sum_of_semicolon + sum_of_coma
        if (sum_of_delimiter / row_count >= 0.25) and multi:
            return []  # keine multinominale
        elif (sum_of_delimiter / row_count < 0.25) and not multi:
            return []  # keine "einfachen" nominale
        split = series.str.split(r"[\W]")
        alltokens = pd.Series(chain.from_iterable(split))
        alltokens = summarize_categoricals(alltokens, official_doc).drop(
            index=["Number of Not Missing Values"]
        )
        alltokens = alltokens.set_axis("Token " + alltokens.index, axis=0)
        return alltokens

    lens = pd.DataFrame({col: data[col].str.len() for col in data})
    lens = summarize_numericals(lens).drop(index=["Number of Not Missing Values"])
    lens = lens.set_axis("Text Length " + lens.index, axis=0)
    cats = summarize_categoricals(data, official_doc)
    # tokens = pd.DataFrame({col: token_analysis(data[col].dropna()) for col in data})#depricated
    # print({col:data for col, data in {col: nominal_analysis(data[col].dropna(), multi=False) for col in data}.items() if len(data) != 0})
    nominals = pd.DataFrame(
        {
            col: data
            for col, data in {
                col: nominal_analysis(data[col].dropna(), multi=False) for col in data
            }.items()
            if len(data) != 0
        }
    )
    multi_nominals = pd.DataFrame(
        {
            col: data
            for col, data in {
                col: nominal_analysis(data[col].dropna(), multi=True) for col in data
            }.items()
            if len(data) != 0
        }
    )
    return [
        pd.concat(
            [
                cats[cats.columns.intersection(nominals.columns)],
                nominals,
                lens[lens.columns.intersection(nominals.columns)],
            ],
            axis=0,
        ),
        pd.concat(
            [
                cats[cats.columns.intersection(multi_nominals.columns)],
                multi_nominals,
                lens[lens.columns.intersection(multi_nominals.columns)],
            ],
            axis=0,
        ),
    ]


def classify_tokens(tokens):
    pass


def summarize(data: pd.DataFrame, limit: int, official_doc: pd.DataFrame = None):
    """Return a dictionary with summarizing information on each column

    Args:
        data (pd.DataFrame): The data
        limit (int): How many values will lead to a column to be considered not a categorical?

    Returns:
        dict: Dictionary with one key for each column data category
    """
    nuniques = data.nunique()
    numTest = data.dtypes.apply(
        lambda dtype: dtype.name == "float64" or dtype.name == "int64"
    )  # sind alle tokens zahlen?
    categoricals_sel = nuniques < limit  # categoricals_sel: weniger tokens als limit
    single_values_sel = nuniques == 1

    new_cat_sel = (
        categoricals_sel & ~numTest
    )  # & ~units_sel #Kategorie = wenig tokens & keine zahl & keine unit

    categoricals = nuniques.index[
        new_cat_sel
    ]  # Um single-values rauszufiltern: & ~single_values_sel
    numerics = nuniques.index[
        ~new_cat_sel & ~single_values_sel & data.apply(is_numeric_dtype)
    ]  # mehr tokens als limit und nummerisch
    strings = nuniques.index[
        ~new_cat_sel
        & ~single_values_sel
        & data.apply(
            lambda col: col.dtype.kind == "O"
        )  # mehr tokens als limit und Datentyp = object
    ]
    single_values = nuniques.index[
        single_values_sel
    ]  # eine konstante einheit (in jeder spalte)

    assert len(numerics) + len(categoricals) + len(strings) + len(single_values) == len(
        data.columns
    )

    if len(categoricals) > 0:
        categoricals = summarize_categoricals(data[categoricals], official_doc)
    else:
        categoricals = None
    if len(numerics) > 0:
        numerics = summarize_numericals(data[numerics], official_doc)
    else:
        numerics = None
    if len(strings) > 0:
        strings = summarize_strings(data[strings], official_doc)
    else:
        strings = None
    return {
        "cat": categoricals,
        "numerics": numerics,
        "nominal": strings[0],
        "multi nominal": strings[1],
    }


def display_data_doc(
    schema=None, data=None, limit=20, columns_only=False, official_doc=None
):
    """Display a formatted Markdown combined information block and schema and data

    Args:
        schema (pa.DataFrameSchema): The schema
        data (pd.DataFrame): The data
        limit (int): How many values will lead to a column to be considered not a categorical?
    """
    if schema is not None:
        res = schema_info(schema)
        display(Markdown(f'**Title:** {res["title"]}'))
        display(Markdown(f'**Description:** {res["description"]}'))
        display(Markdown("Index:\n"))
        display(HTML(res["index"].to_html(escape=False)))
        display(Markdown("Columns:\n"))
        display(HTML(res["columns"].to_html(escape=False)))
        display(Markdown("Further Checks:\n\nNone"))
    if data is not None:
        sums = summarize(data, limit, official_doc)
        if not columns_only:
            display(
                Markdown(
                    f"The data has {data.shape[0]} rows and {data.shape[1]} columns."
                )
            )
        if sums["cat"] is not None:
            display(
                Markdown(
                    f'Summary statistics for the {sums["cat"].shape[1]} columns, which contained categorical data are shown in the next table. Categorical data was defined as having less than {limit} distinct values.\n'
                )
            )
            display(sums["cat"])
        if sums["numerics"] is not None:
            display(
                Markdown(
                    f'The next table shows summary statistics for the {sums["numerics"].shape[1]} columns with numerical data. \n'
                )
            )
            display(sums["numerics"])
        if sums["nominal"] is not None:
            display(
                Markdown(
                    f'This table shows the {sums["nominal"].shape[1]} columns which contain nominal data.\n'
                )
            )
            display(sums["nominal"])
        if sums["multi nominal"] is not None:
            display(
                Markdown(
                    f'This final table shows the {sums["multi nominal"].shape[1]} columns with multi nominal data.\n'
                )
            )
            display(sums["multi nominal"])


# als apply axis=0
def check_for_single_values(col):
    filtered = col.dropna()
    no_of_values = filtered.nunique()

    if no_of_values == 1:
        return True
    return False

In [4]:
examples = [
    "../../../resources/input/element_empfaenger.csv",
    "../../../resources/input/element_empfaenger_dringlichkeit.csv",
    "../../../resources/input/element_spender_postmortem_labor_hla.csv",
    "../../../resources/input/element_empfaenger_immunologie.csv",
    "../../../resources/input/element_transplantation.csv",
]

examples = [examples[-1]]
examples = [pd.read_csv(f, sep=";", low_memory=False) for f in examples]
offical_doc = pd.read_csv("../../../results/officialdoc.csv")
iter(examples)

In [19]:
len(examples[0].columns[examples[0].apply(check_for_single_values)])
len(examples[0].columns)

156

In [14]:
x = (~examples[0].isna()).sum(axis=0) == 0
print(x.index[x == True])
print(x.value_counts())
print(examples[0]["TBlutungenNPIQTIG"].value_counts())

Index(['TBiopsieET', 'TFlugtransportOrganET', 'TPostOPAbstossungDateET',
       'TPostOPAbstossungMedikamentoesBehandeltET',
       'TPostOPAbstossungOrgan1LuET', 'TPostOPAbstossungOrgan2LuET',
       'TPostOPFunktionGeraeteMechanischET',
       'TPostOPKomplikationAtemwegeLuET'],
      dtype='object')
False    148
True       8
Name: count, dtype: int64
TBlutungenNPIQTIG
ja    1351
Name: count, dtype: int64


In [9]:
# pr = cProfile.Profile()
# pr.enable()

display_data_doc(data=examples[0], official_doc=offical_doc)
# pr.disable()

# pr.dump_stats('prof1.prof')

TBlutungenNPIQTIG
ja    1351
Name: count, dtype: int64


The data has 125044 rows and 156 columns.

Summary statistics for the 81 columns, which contained categorical data are shown in the next table. Categorical data was defined as having less than 20 distinct values.


,TAbbruchTxIQTIG,TAzathioprinHGabeIQTIG,TAzathioprinLuGabeIQTIG,TBestimmungsortET,TBetriebsstaettennummerIQTIG,TCyclosporinHGabeIQTIG,TCyclosporinLuGabeIQTIG,TDrainagePgangET,TDrainageVenenET,TDringlET,TDringlHIQTIG,TDringlLeIQTIG,TDringlLuIQTIG,TEinzelOderDoppelTransplantationNIQTIG,TEntlassungAzathioprinHGabeIQTIG,TEntlassungAzathioprinLuGabeIQTIG,TEntlassungCyclosporinHGabeIQTIG,TEntlassungCyclosporinLuGabeIQTIG,TEntlassungFunktionTransplantatNIQTIG,TEntlassungHunterstuetzungssystemKunstherzHIQTIG,TEntlassungImmunsuppressivaAndereHGabeIQTIG,TEntlassungImmunsuppressivaAndereLuGabeIQTIG,TEntlassungInsulinfreiPIQTIG,TEntlassungMToRHemmerHGabeIQTIG,TEntlassungMToRHemmerLuGabeIQTIG,TEntlassungMycophenolatHGabeIQTIG,TEntlassungMycophenolatLuGabeIQTIG,TEntlassungSteroideHGabeIQTIG,TEntlassungSteroideLuGabeIQTIG,TEntlassungTacrolimusHGabeIQTIG,TEntlassungTacrolimusLuGabeIQTIG,TEntlassungTracheotomieLuIQTIG,TEntnahmeTransplantatPIQTIG,TEntnahmeTransplantatUrsachePIQTIG,TFollowUpLostToFollowUpET,THypotensivePeriodeHIQTIG,TImmunsuppression1ET,TImmunsuppression3ET,TImmunsuppression4ET,TImmunsuppressivaAndereHGabeIQTIG,TImmunsuppressivaAndereLuGabeIQTIG,TImplantationsstelleET,TInduktionstherapieHIQTIG,TInduktionstherapieLuIQTIG,TKatecholamintherapieHIQTIG,TKomplikationIntraPostOperationAllgmeinNPIQTIG,TMToRInhibitorHGabeIQTIG,TMToRInhibitorLuGabeIQTIG,TMycophenolatHGabeIQTIG,TMycophenolatLuGabeIQTIG,TOCSSystemHIQTIG,TOperationSimultanLuIQTIG,TOrganET,TOrganfunktionInitialET,TOrganqualitaetHIQTIG,TOrganteilLeIQTIG,TPostOPAbstossungNIQTIG,TPostOPAbstossungPIQTIG,TPostOPFunktionsaufnahmeTransplantatNIQTIG,TPostOPTodesursacheHIQTIG,TPostOPTodesursacheLuIQTIG,TPostOPTodesursacheNIQTIG,TRelaparotomieNPIQTIG,TRelaparotomieUrsacheNPIQTIG,TRetransplantationLuIQTIG,TRetransplantationNIQTIG,TRetransplantationPIQTIG,TSpendeKompatibelNPIQTIG,TSteroideHGabeIQTIG,TSteroideLuGabeIQTIG,TStillstandHIQTIG,TTacrolimusHGabeIQTIG,TTacrolimusLuGabeIQTIG,TTransplantationArtLuIQTIG,TTransplantationArtPNIQTIG,TTransplantationPartiellET,TTransplantationTechnikET,TTransplantationTechnikPET,TVergabeProgramET,TVergabeTypET,TZentrumsangebotLeIQTIG
Info,Abbruch der Transplantation,Azathioprin,Azathioprin,Transplant Destination,Betriebsstätten-Nummer,Cyclosporin,Cyclosporin,Pancreatic duct drainage,Venous drainage,Urgency at Transplantation,Dringlichkeit,Dringlichkeit der Transplantation gemäß Medica...,Dringlichkeit,Einzel- oder Doppeltransplantation bei isolier...,Azathioprin,Azathioprin,Cyclosporin,Cyclosporin,funktionierendes Nierentransplantat bei Entlas...,Wurde der Patient mit einem Herzunterstützungs...,andere,andere,Patient bei Entlassung insulinfrei?,m-ToR-Inhibitor,m-ToR-Inhibitor,Mycophenolat,Mycophenolat,Steroide,Steroide,Tacrolimus,Tacrolimus,Patient bei Entlassung tracheotomiert,Entnahme des Pankreastransplantats erforderlich,Ursache für die Entnahme des Pankreastransplan...,Out of analysis,hypotensive Periode,immunosuppression,immunosuppression,immunosuppression,andere,andere,Site of implantation,Induktionstherapie,Induktionstherapie,Katecholamintherapie,behandlungsbedürftige (schwere) intra- oder po...,m-ToR-Inhibitor,m-ToR-Inhibitor,Mycophenolat,Mycophenolat,Einsatz des Organ Care System (OCS),simultane Operationen,Organ Transplanted,Immediate performance,Organqualität zum Zeitpunkt der Transplantation,Spenderorgan,akute behandlungsbedürftige Rejektion Niere,akute behandlungsbedürftige Rejektion Pankreas,Postoperative Funktionsaufnahme des Transplantats,Todesursache(n) akut,Todesursache(n) akut,Todesursache,Relaparotomie erforderlich,Ursache für die Relaparotomie,Retransplantation,Retransplantation Niere,Retransplantation Pankreas,Kompatible Spende,Steroide,Steroide,Herzstillstand,Tacrolimus,Tacrolimus,Transplantationsart,durchgeführte Transplantation,Partial Transplant,Transplant technique,Pancreas transplant technique,Allocation program,Allocation type,Zentrumsangebot
Content,"Auswahlliste: ""nein"", ""ja""","Auswahlliste: ""nein"", ""ja""","Auswahl

The next table shows summary statistics for the 35 columns with numerical data. 


,TAufnahmeKrankenhausDateIQTIG,TAufnahmeWartelisteDateET,TBiopsieET,TCreatinkinaseHWertIQTIG,TCreatinkinaseMBHWertIQTIG,TEntlassungFEV1LuWertIQTIG,TEntlassungGrundIQTIG,TEntlassungKrankenhausDateIQTIG,TEntlassungStandortIQTIG,TFachabteilungIQTIG,TFlugtransportOrganET,TFollowUpLetztesDateET,THaematokritHWertIQTIG,TIschaemiezeitGesamtLuWertIQTIG,TIschaemiezeitKaltHWertIQTIG,TIschaemiezeitKaltLeWertIQTIG,TIschaemiezeitKaltWertET,TIschaemiezeitWarmZweiteWertET,TPostOPAbstossungDateET,TPostOPAbstossungHBehandeltAnzahlIQTIG,TPostOPAbstossungMedikamentoesBehandeltET,TPostOPAbstossungOrgan1LuET,TPostOPAbstossungOrgan2LuET,TPostOPAnzahlDialysenNIQTIG,TPostOPFunktionGeraeteMechanischET,TPostOPHarnausscheidungErsteStundeWertET,TPostOPHarnausscheidungMengeRestWertET,TPostOPHarnausscheidungMengeWertET,TPostOPKomplikationAtemwegeLuET,TPostOPKreatininNWertIQTIG,TPostOPOrganversagenDateET,TPostOPOrganversagenExplantationDateET,TTxDateET,TTxDateIQTIG,TTxZeitpunktET
Info,Aufnahmedatum Krankenhaus,Waiting since,Biopsy,CK-Wert,CK-MB-Wert,FEV1 (prädiktiver Wert in %),Entlassungsgrund,Entlassungsdatum Krankenhaus,entlassender Standort,Fachabteilung,Organ Transported by Plane,Date last seen,Hämatokrit (Hk),Gesamtischämiezeit,kalte Ischämiezeit,kalte Ischämiezeit (Stunden) bzw. kalte Ischäm...,Cold ischaemic period,Warm ischaemic period 2,Rejection date,Anzahl der behandelten Abstoßungsreaktionen,Any drug-treated rejection during the hospital...,Rejection Organ 1,Rejection Organ 2,Anzahl postoperativer Dialysen bis Funktionsau...,Any mechanical device immediately post-operati...,Diuresis first hour,Rest diuresis ml,Diuresis ml,Lung transplant: airway complications,"Kreatininwert i.S. in mg/dl,Kreatininwert i.S....",Date of failure,Date of explantation on failure,Date of Transplant,"OP-Datum,OP-Datum,Datum der Transplantation,OP...",Time of Transplant
Content,Ganzzahl,Zeitstempel (YYYY-MM-DDThh:mm:ss),"Auswahlliste: ""Yes"", ""No""",Ganzzahl,Ganzzahl,Dezimalzahl,Zeichenkette,Zeitstempel (YYYY-MM-DDThh:mm:ss),Zeichenkette,Zeichenkette,"Auswahlliste: ""Yes"", ""No""",Zeitstempel (YYYY-MM-DDThh:mm:ss),Ganzzahl,Ganzzahl,Ganzzahl,Ganzzahl,Dezimalzahl,Dezimalzahl,Zeitstempel (YYYY-MM-DDThh:mm:ss),Ganzzahl,"Auswahlliste: ""Yes"", ""No""","Auswahlliste: ""Yes"", ""No""","Auswahlliste: ""Yes"", ""No""",Ganzzahl,"Auswahlliste: ""Yes"", ""No""",Dezimalzahl,Dezimalzahl,Dezimalzahl,"Auswahlliste: ""Yes"", ""No""",Dezimalzahl,Zeitstempel (YYYY-MM-DDThh:mm:ss),Zeitstempel (YYYY-MM-DDThh:mm:ss),Zeitstempel (YYYY-MM-DDThh:mm:ss),Zeitstempel (YYYY-MM-DDThh:mm:ss),Zeitstempel (YYYY-MM-DDThh:mm:ss)
Number of Not Missing Values,36601.0,52029.0,0.0,1182.0,2060.0,1828.0,46536.0,37013.0,11489.0,46536.0,0.0,38472.0,864.0,3397.0,23878.0,5821.0,42037.0,32028.0,0.0,1364.0,0.0,0.0,0.0,16606.0,0.0,12115.0,17591.0,16889.0,0.0,5566.0,7294.0,1686.0,52029.0,26475.0,52029.0
Mean,3791.648507,2591.935805,NaN,877.42555,47.479612,452.853556,2.946815,3819.347364,0.063887,1609.36187,NaN,4982.879939,27.390046,440.191051,562.8409,542.740079,546.826296,38.678282,NaN,0.27566,NaN,NaN,NaN,0.863905,NaN,160.664466,23.621227,2724.150157,NaN,165.447898,3788.97875,3682.723013,3342.754541,4158.228026,3342.754541
Std,941.957411,1478.972772,NaN,3805.22203,115.033948,899.950958,3.207998,945.431628,0.288641,619.263078,NaN,3371.731897,11.580418,185.141049,384.012303,195.363763,326.460732,48.062111,NaN,0.584937,NaN,NaN,NaN,2.715923,NaN,284.348058,2.283082,3554.874546,NaN,88.833505,1244.552288,8855.767968,1131.766409,708.530399,1131.766409
Min,2153.0,-4872.0,NaN,0.0,0.0,0.0,1.0,2156.0,0.0,100.0,NaN,1430.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,NaN,0.0,812.0,1441.0,1423.0,2884.0,1423.0
25%,3012.0,1617.0,NaN,93.0,13.0,60.0,1.0,3028.0,0.0,1500.0,NaN,4134.0,24.0,337.0,240.0,443.0,290.0,27.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,24.0,510.0,NaN,113.0,2831.25,2463.0,2373.0,3559.0,2373.0
Median,3725.0,2644.0,NaN,241.5,24.0,421.5,1.0,3749.0,0.0,1500.0,NaN,5486.0,29.0,450.0,550.0,547.0,527.0,35.0,NaN,

This table shows the 9 columns which contain nominal data.


,TEntlassungDiagnoseICD10BeschreibungIQTIG,TEntlassungDiagnoseICD10IQTIG,TImmunsuppressionInitial1ET,TImmunsuppressionInitial2ET,TImmunsuppressionInitial3ET,TImmunsuppressionInitial4ET,TOPSCodeBeschreibungIQTIG,TOPSCodeIQTIG,TPostOPOrganversagenUrsacheET
Info,Entlassungsdiagnose(n),Entlassungsdiagnose(n),immunosuppression,immunosuppression,immunosuppression,immunosuppression,Operation,Operation,Cause of failure
Content,Zeichenkette,Zeichenkette,"Auswahlliste: ""None"", ""Corticosteroids"", ""Azat...","Auswahlliste: ""None"", ""Corticosteroids"", ""Azat...","Auswahlliste: ""None"", ""Corticosteroids"", ""Azat...","Auswahlliste: ""None"", ""Corticosteroids"", ""Azat...",Zeichenkette,Zeichenkette,Zeichenkette
Number of Not Missing Values,43034 (34.42 %),43034 (34.42 %),24602 (19.67 %),24365 (19.49 %),22037 (17.62 %),10850 (8.68 %),50415 (40.32 %),50415 (40.32 %),6673 (5.34 %)
Number of Distinct Values (Ignoring Missing Values),29679 (23.73 %),29679 (23.73 %),21 (0.02 %),25 (0.02 %),25 (0.02 %),24 (0.02 %),13379 (10.70 %),13379 (10.70 %),67 (0.05 %)
Most Common Value,'N18.0' (3235),'N18.0' (3235),'Tacrolimus (FK-506)' (7659),'MMF-Cellcept' (9147),'Corticosteroids' (7796),'Corticosteroids' (4683),'5-555.1' (6681),'5-555.1' (6681),'Rejection while taking immunosuppressive drug...
Least Common Value,"'N18.5,J18.9,Z94.0,J16.8,J18.8,N18.4,I77.1' (1)","'N18.5,J18.9,Z94.0,J16.8,J18.8,N18.4,I77.1' (1)",'Mizoribine' (2),'MoAb to epitopes' (1),'Methotrexate' (1),'ALG' (1),"'5-555.10,5-554.71,5-984' (1)","'5-555.10,5-554.71,5-984' (1)",'Constrictive / Restrictive disease ( heart ) ...
All distinct Values,29679 unique values,29679 unique values,21 unique values,25 unique values,25 unique values,24 unique values,13379 unique values,13379 unique values,67 unique values
Token Info,NA,NA,NA,NA,NA,NA,NA,NA,NA
Token Content,NA,NA,NA,NA,NA,NA,NA,NA,NA
Token Number of Distinct Values (Ignoring Missing Values),1400 (0.34 %),1400 (0.34 %),30 (0.04 %),34 (0.06 %),37 (0.08 %),36 (0.18 %),923 (0.29 %),923 (0.29 %),120 (0.20 %)


This final table shows the 12 columns with multi nominal data.


,TEntlassungDiagnoseELTRLeIQTIG,TFollowUpZentrumET,TIdEmpfaengerNrETET,TIdEmpfaengerNrETIQTIG,TIdSpenderNrETET,TIdSpenderNrETIQTIG,TIdTransplantationsnummerETET,TImmunsuppression2ET,TInstitutionskennzeichenIQTIG,TPostOPTodesursacheLeIQTIG,TTransplantationszentrumET,TTransplantationszentrumRegistrierungET
Info,Entlassungsdiagnose nach ELTR,Follow-up Center,Recipient number,Empfänger ID,Donor Registration Number,Spender ID,Transplant Number,immunosuppression,Institutionskennzeichen,Todesursache,Transplant at Center,Transplant registration center
Content,Zeichenkette,Zeichenkette,Zeichenkette,Zeichenkette,Zeichenkette,Zeichenkette,Zeichenkette,"Auswahlliste: ""None"", ""Corticosteroids"", ""Azat...",Zeichenkette,Zeichenkette,Zeichenkette,Zeichenkette
Number of Not Missing Values,1182 (0.95 %),52027 (41.61 %),52029 (41.61 %),71854 (57.46 %),52029 (41.61 %),38118 (30.48 %),52029 (41.61 %),7688 (6.15 %),46536 (37.22 %),1630 (1.30 %),52029 (41.61 %),52029 (41.61 %)
Number of Distinct Values (Ignoring Missing Values),61 (0.05 %),60 (0.05 %),44634 (35.69 %),42696 (34.14 %),23669 (18.93 %),21096 (16.87 %),52029 (41.61 %),20 (0.02 %),76 (0.06 %),41 (0.03 %),47 (0.04 %),47 (0.04 %)
Most Common Value,'E1' (301),'IwVkn8rVsieZQ6vNkACcEg==' (5960),'BxcI12vHZaJpbBJWsMspjA==' (6),'4SdvJLTeqRBbL8ejMF6vgg==' (1844),'sO2glqbVcMGYlOqF5TpjcQ==' (9),'NCYHuHMbzhK5FTrSC45vww==' (9),'DrQCaNeJUCPH9ET0yBUnig==' (1),'MMF-Cellcept' (3583),'9f5ZvZAX7n0AaEavqox9ug==' (4329),'B1' (359),'IwVkn8rVsieZQ6vNkACcEg==' (6017),'IwVkn8rVsieZQ6vNkACcEg==' (6017)
Least Common Value,'A15' (1),'c4XYlc4zvLqgjvy/qd+Urg==' (1),'ZQJ7oclf5BxAMuKP/EerBQ==' (1),'cgS9QkKLi0GamOQx7R0rpw==' (1),'glZcM+p4kJPXOzrGugU50Q==' (1),'UwQip2/Ern4YMS+l+VQZjQ==' (1),'KheQ9k2hgmRFiGG+RQrckg==' (1),'Methotrexate' (1),'m+l6sZNJP/zbYgeP2GgJcw==' (1),'C17' (1),'g7Un/e3Lg5RDryRJNjqm6g==' (19),'g7Un/e3Lg5RDryRJNjqm6g==' (19)
All distinct Values,61 unique values,60 unique values,44634 unique values,42696 unique values,23669 unique values,21096 unique values,52029 unique values,20 unique values,76 unique values,41 unique values,47 unique values,47 unique values
Token Info,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Token Content,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
Token Number of Distinct Values (Ignoring Missing Values),61 (5.16 %),101 (0.05 %),66343 (34.87 %),63469 (24.26 %),35551 (18.67 %),31707 (22.71 %),77340 (40.64 %),28 (0.18 %),113 (0.07 %),41 (2.52 %),80 (0.04 %),80 (0.04 %)


In [5]:
display_data_doc(data=examples[1])

<class 'pandas.core.indexes.base.Index'>
<class 'pandas.core.indexes.base.Index'>


The data has 307832 rows and 5 columns.

Summary statistics for the 2 columns, which contained categorical data are shown in the next table. Categorical data was defined as having less than 20 distinct values.


,EDringlCodeET,EDringlOrganET
Number of Not Missing Values,307832 (100.00 %),307832 (100.00 %)
Number of Distinct Values (Ignoring Missing Values),16 (0.01 %),6 (0.00 %)
Most Common Value,'NT - Not Transplantable' (79555),'Ki' (192174)
Least Common Value,'1A HU - Unstable on inotropes' (2),'In' (122)
All distinct Values,16 unique values,6 unique values


The next table shows summary statistics for the 1 columns with numerical data. 


,EDringlDateET
Number of Not Missing Values,307832.000000
Mean,3319.496787
Std,1263.829064
Min,-2960.000000
25%,2306.000000
Median,3339.000000
75%,4385.000000
Max,5440.000000


This table shows the 1 columns which contain nominal data.


,EDringlGrundET
Number of Not Missing Values,146707 (47.66 %)
Number of Distinct Values (Ignoring Missing Values),29 (0.01 %)
Most Common Value,'Transplanted' (46801)
Least Common Value,'(Uremic) Polyneuropathy' (4)
All distinct Values,29 unique values
Token Number of Distinct Values (Ignoring Missing Values),95 (0.02 %)
Token Most Common Value,'' (133855)
Token Least Common Value,'Uremic' (4)
Token All distinct Values,95 unique values
Text Length Mean,21.73719


This final table shows the 1 columns with multi nominal data.


,EDringlIdEmpfaengerNrETET
Number of Not Missing Values,307832 (100.00 %)
Number of Distinct Values (Ignoring Missing Values),78641 (25.55 %)
Most Common Value,'s6pYHJxtSFOwYB4UfVhrhQ==' (68)
Least Common Value,'1kQYijGFG1K7XpeZpQGALA==' (1)
All distinct Values,78641 unique values
Token Number of Distinct Values (Ignoring Missing Values),116177 (10.32 %)
Token Most Common Value,'' (631361)
Token Least Common Value,'j3kofHGYr7M2aaVgCYiVgg' (1)
Token All distinct Values,116177 unique values
Text Length Mean,24.0


In [6]:
display_data_doc(data=examples[2])

<class 'pandas.core.indexes.base.Index'>
<class 'pandas.core.indexes.base.Index'>


The data has 42269 rows and 54 columns.

Summary statistics for the 30 columns, which contained categorical data are shown in the next table. Categorical data was defined as having less than 20 distinct values.


,SPostmLaborHLAABR1ETDSO,SPostmLaborHLAABR2ETDSO,SPostmLaborHLAASP1ETDSO,SPostmLaborHLAASP2ETDSO,SPostmLaborHLABW4DSO,SPostmLaborHLABW6DSO,SPostmLaborHLACBR1ETDSO,SPostmLaborHLACBR2ETDSO,SPostmLaborHLACDNA1DSO,SPostmLaborHLACDNA2DSO,...,SPostmLaborHLADRB3DSO,SPostmLaborHLADRB4DSO,SPostmLaborHLADRB5DSO,SPostmLaborHLADRBR1ETDSO,SPostmLaborHLADRBR2ETDSO,SPostmLaborHLADRSP1ETDSO,SPostmLaborHLADRSP2ETDSO,SPostmLaborHLAProbenmaterialDSO,SPostmLaborHLATypisierungDNAET,SPostmLaborHLATypisierungsgewebeET
Number of Not Missing Values,12216 (28.90 %),10744 (25.42 %),1476 (3.49 %),6758 (15.99 %),10186 (24.10 %),10234 (24.21 %),8596 (20.34 %),7129 (16.87 %),6537 (15.47 %),5732 (13.56 %),...,10324 (24.42 %),10276 (24.31 %),10195 (24.12 %),12209 (28.88 %),11138 (26.35 %),3640 (8.61 %),3640 (8.61 %),12287 (29.07 %),30 (0.07 %),29582 (69.99 %)
Number of Distinct Values (Ignoring Missing Values),9 (0.02 %),10 (0.02 %),13 (0.03 %),14 (0.03 %),3 (0.01 %),3 (0.01 %),14 (0.03 %),14 (0.03 %),17 (0.04 %),17 (0.04 %),...,3 (0.01 %),3 (0.01 %),3 (0.01 %),10 (0.02 %),10 (0.02 %),7 (0.02 %),7 (0.02 %),9 (0.02 %),2 (0.00 %),5 (0.01 %)
Most Common Value,'A2' (5071),'A19' (2612),'A24' (620),'A24' (1558),'Pos' (6402),'Pos' (8743),'Cw3' (1967),'Cw7' (2832),'Cw*07' (1440),'Cw*07' (2049),...,'Pos' (6782),'Neg' (5452),'Neg' (7002),'DR1' (2545),'DR2' (3014),'DR11' (1206),'DR11' (1206),'Blut' (11973),'SSO' (22),'Peripheral blood' (29400)
Least Common Value,'A36' (3),'A80' (2),'A34' (1),'A34' (18),'NT' (40),'NT' (38),'Cw18' (1),'Cw18' (8),'Cw*18' (1),'Cw*0302' (6),...,'NT' (16),'NT' (20),'NT' (25),'DR9' (97),'DR1' (71),'DR16' (104),'DR16' (104),'Gewebe' (1),'SSP' (8),'Other' (16)
All distinct Values,9 unique values,10 unique values,13 unique values,14 unique values,"""NT"", ""Neg"", ""Pos""","""NT"", ""Pos"", ""Neg""",14 unique values,14 unique values,17 unique values,17 unique values,...,"""Pos"", ""Neg"", ""NT""","""Neg"", ""Pos"", ""NT""","""Neg"", ""Pos"", ""NT""",10 unique values,10 unique values,7 unique values,7 unique values,9 unique values,"""SSO"", ""SSP""","""Peripheral blood"", ""Spleen"", ""Lymph nodes"", ""..."


The next table shows summary statistics for the 5 columns with numerical data. 


,SPostmLaborHLABefundDateDSO,SPostmLaborHLAEintragDateET,SPostmLaborHLAErfahrenAmDateDSO,SPostmLaborHLAProbeDateDSO,SPostmLaborHLAUntersuchungDateDSO
Number of Not Missing Values,12213.000000,29982.000000,12228.000000,12287.00000,6.218000e+03
Mean,3262.318677,3322.812087,3261.814197,3257.32628,3.478218e+03
Std,1131.173354,1147.233048,1131.542152,1132.97306,1.626997e+04
Min,1424.000000,-499.000000,1424.000000,-67.00000,-1.428100e+04
25%,2296.000000,2343.000000,2295.500000,2289.00000,2.157250e+03
Median,3180.000000,3254.000000,3179.000000,3175.00000,2.995500e+03
75%,4171.000000,4277.750000,4171.000000,4168.00000,4.028750e+03
Max,5451.000000,6138.000000,5451.000000,5444.00000,1.168500e+06


This table shows the 1 columns which contain nominal data.


,SPostmLaborHLAAntigeneET
Number of Not Missing Values,29982 (70.93 %)
Number of Distinct Values (Ignoring Missing Values),13917 (32.92 %)
Most Common Value,'A1 A3 B7 B8 Bw6 Cw7 DR2 DR15 DR3 DR51 DR52 DQ...
Least Common Value,'A2 A19 A29 B12 B44 Bw4 Cw5 Cw16 DR2 DR15 DR7 ...
All distinct Values,13917 unique values
Token Number of Distinct Values (Ignoring Missing Values),150 (0.03 %)
Token Most Common Value,'Bw6' (21752)
Token Least Common Value,'12' (2)
Token All distinct Values,150 unique values
Text Length Mean,57.702955


This final table shows the 18 columns with multi nominal data.


,SPostmLaborHLAADNA1DSO,SPostmLaborHLAADNA2DSO,SPostmLaborHLAASER1DSO,SPostmLaborHLAASER2DSO,SPostmLaborHLABBR1ETDSO,SPostmLaborHLABBR2ETDSO,SPostmLaborHLABDNA1DSO,SPostmLaborHLABDNA2DSO,SPostmLaborHLABSER1DSO,SPostmLaborHLABSER2DSO,SPostmLaborHLABSP1ETDSO,SPostmLaborHLABSP2ETDSO,SPostmLaborHLADRDNA1DSO,SPostmLaborHLADRDNA2DSO,SPostmLaborHLAIdDSOKennnummerDSO,SPostmLaborHLAIdSpenderNrETDSO,SPostmLaborHLAIdSpenderNrETET,SPostmLaborHLATypisierungszentrumET
Number of Not Missing Values,11999 (28.39 %),10551 (24.96 %),8719 (20.63 %),7494 (17.73 %),12212 (28.89 %),11486 (27.17 %),11998 (28.38 %),11270 (26.66 %),8873 (20.99 %),8256 (19.53 %),3493 (8.26 %),7922 (18.74 %),12209 (28.88 %),11138 (26.35 %),12287 (29.07 %),12287 (29.07 %),29982 (70.93 %),29982 (70.93 %)
Number of Distinct Values (Ignoring Missing Values),22 (0.05 %),26 (0.06 %),21 (0.05 %),23 (0.05 %),23 (0.05 %),28 (0.07 %),48 (0.11 %),61 (0.14 %),44 (0.10 %),47 (0.11 %),21 (0.05 %),22 (0.05 %),22 (0.05 %),28 (0.07 %),12248 (28.98 %),12248 (28.98 %),13681 (32.37 %),40 (0.09 %)
Most Common Value,'A*02' (4975),'A*03' (1672),'A2' (3641),'A3' (1166),'B7' (2918),'B12' (1973),'B*07' (2860),'B*44' (1860),'B7' (2095),'B44' (1230),'B44' (1030),'B44' (1883),'DRB1*01' (2512),'DRB1*15' (2464),'m0AGr++miGUhmHIdyxtYKQ==' (3),'P6fXpUsME0TQecjX3EzHow==' (3),'mWDDog6mlO9VsZsyUNwLbA==' (10),'l6cB2k2uaG8ERdskpcMPoA==' (3412)
Least Common Value,'A*34' (1),'A*3201' (1),'A34' (1),'A80' (2),'B46' (1),'B82' (1),'B*5002' (1),'B*1509' (1),'B22' (1),'B78' (1),'B75' (4),'B54' (3),'DRB1*1601' (1),'DRB1*1401' (1),'q2VqcEn6Qc9tgEfxwQA+rA==' (1),'//l2RBS86NkgfxbYcgzNVg==' (1),'y6ynJYUArkFjNpB9CRZQJg==' (2),'t7Z6aAs6yAjklPR6SzS4xA==' (2)
All distinct Values,22 unique values,26 unique values,21 unique values,23 unique values,23 unique values,28 unique values,48 unique values,61 unique values,44 unique values,47 unique values,21 unique values,22 unique values,22 unique values,28 unique values,12248 unique values,12248 unique values,13681 unique values,40 unique values
Token Number of Distinct Values (Ignoring Missing Values),23 (0.10 %),27 (0.13 %),21 (0.24 %),23 (0.31 %),23 (0.19 %),28 (0.24 %),49 (0.20 %),62 (0.28 %),44 (0.50 %),47 (0.57 %),21 (0.60 %),22 (0.28 %),23 (0.09 %),29 (0.13 %),18464 (41.17 %),18505 (41.18 %),20675 (18.84 %),70 (0.07 %)
Token Most Common Value,'A' (11999),'A' (10551),'A2' (3641),'A3' (1166),'B7' (2918),'B12' (1973),'B' (11998),'B' (11270),'B7' (2095),'B44' (1230),'B44' (1030),'B44' (1883),'DRB1' (12209),'DRB1' (11138),'' (25153),'' (25222),'' (61568),'' (60272)
Token Least Common Value,'34' (1),'3201' (1),'A34' (1),'A80' (2),'B46' (1),'B82' (1),'5002' (1),'1509' (1),'B22' (1),'B78' (1),'B75' (4),'B54' (3),'1601' (1),'1401' (1),'q2VqcEn6Qc9tgEfxwQA' (1),'l2RBS86NkgfxbYcgzNVg' (1),'y6ynJYUArkFjNpB9CRZQJg' (2),'ct3foKVJijS0eqswg' (2)
Token All distinct Values,23 unique values,27 unique values,21 unique values,23 unique values,23 unique values,28 unique values,49 unique values,62 unique values,44 unique values,47 unique values,21 unique values,22 unique values,23 unique values,29 unique values,18464 unique values,18505 unique values,20675 unique values,70 unique values
Text Length Mean,4.000667,4.001706,2.162977,2.717107,2.570914,2.845116,4.068511,4.092813,2.60408,2.959908,3.0,3.0,7.094848,7.038966,24.0,24.0,24.0,24.0


In [7]:
display_data_doc(data=examples[3])

<class 'pandas.core.indexes.base.Index'>
<class 'pandas.core.indexes.base.Index'>


The data has 635400 rows and 20 columns.

Summary statistics for the 5 columns, which contained categorical data are shown in the next table. Categorical data was defined as having less than 20 distinct values.


,EImmAntikoerperNichtZytotoxischET,EImmAutoantikoerperET,EImmCrossMatchDTTET,EImmDiagnostikScreeningverfahrenET,EImmErgebnisTypET
Number of Not Missing Values,15178 (2.39 %),563978 (88.76 %),131770 (20.74 %),563658 (88.71 %),635400 (100.00 %)
Number of Distinct Values (Ignoring Missing Values),2 (0.00 %),3 (0.00 %),2 (0.00 %),6 (0.00 %),4 (0.00 %)
Most Common Value,'Yes' (14230),'Not Tested' (518802),'No' (80372),'CDC' (222483),'Antibody Screening' (563978)
Least Common Value,'No' (948),'Positive' (8704),'Yes' (51398),'Virtual PRA' (20271),'Acceptable Test' (1548)
All distinct Values,"""Yes"", ""No""","""Not Tested"", ""Negative"", ""Positive""","""No"", ""Yes""",6 unique values,"""Antibody Screening"", ""HLA Typing"", ""Unaccepta..."


The next table shows summary statistics for the 5 columns with numerical data. 


,EImmAntikoerperET,EImmEingabeDateET,EImmPRAWertET,EImmProbeDateET,EImmvPRAWertET
Number of Not Missing Values,0.0,635400.000000,563978.000000,635400.000000,13335.000000
Mean,NaN,3417.231069,7.680131,3355.659098,65.633897
Std,NaN,1246.811605,21.099928,1283.751905,30.231034
Min,NaN,-4824.000000,0.000000,-23466.000000,0.000000
25%,NaN,2378.000000,0.000000,2330.000000,43.830000
Median,NaN,3471.000000,0.000000,3408.000000,72.840000
75%,NaN,4494.000000,0.000000,4444.000000,93.420000
Max,NaN,5440.000000,100.000000,5478.000000,100.000000


This table shows the 6 columns which contain nominal data.


,EImmAntigenAkzeptabelET,EImmAntigenInakzeptabelET,EImmDonorFrequencyETKASET,EImmDonorFrequencyHET,EImmHLAPhaenotypisierungET,EImmSpezifitaetenET
Number of Not Missing Values,1537 (0.24 %),24192 (3.81 %),1457 (0.23 %),1457 (0.23 %),43302 (6.81 %),57113 (8.99 %)
Number of Distinct Values (Ignoring Missing Values),1531 (0.24 %),16893 (2.66 %),905 (0.14 %),994 (0.16 %),38248 (6.02 %),18760 (2.95 %)
Most Common Value,'A25 A34 A66 A30 A31 A32 A33 A36 A43 A80 B52 B...,'A2' (352),"'0,000' (44)","'40,975' (37)",'DQ1 DQ6 DQ3 DQ7' (61),'A2' (3350)
Least Common Value,'A9 A23 A24 A10 A25 A26 A34 A66 A11 A19 A29 A3...,'A24 A25 B27 DQ2' (1),"'11,710' (1)","'11,710' (1)",'A9 A24 A10 A26 B7 B40 B60' (1),'A1 A25 A26 A34 A11 A36 A43 A80 B8 B73 Cw6 Cw7...
All distinct Values,1531 unique values,16893 unique values,905 unique values,994 unique values,38248 unique values,18760 unique values
Token Number of Distinct Values (Ignoring Missing Values),127 (0.29 %),127 (0.05 %),491 (16.85 %),554 (19.01 %),195 (0.03 %),128 (0.03 %)
Token Most Common Value,'B64' (785),'A2' (5790),'0' (284),'0' (210),'DQ1' (26952),'A2' (12522)
Token Least Common Value,'Bw4' (1),'Cw13' (1),'948' (1),'517' (1),'80' (1),'Cw13' (1)
Token All distinct Values,127 unique values,127 unique values,491 unique values,554 unique values,195 unique values,128 unique values
Text Length Mean,116.409239,39.761037,5.515443,5.652025,56.677382,25.035333


This final table shows the 2 columns with multi nominal data.


,EImmDiagnostikZentrumET,EImmIdEmpfaengerNrETET
Number of Not Missing Values,635033 (99.94 %),635400 (100.00 %)
Number of Distinct Values (Ignoring Missing Values),58 (0.01 %),54453 (8.57 %)
Most Common Value,'lwkqA/aMYSxPv1pWPoK3zw==' (82284),'MNvooeELJKr5cxN8/SGAgg==' (114)
Least Common Value,'1enebfMvU7ny/SSki+tg6g==' (1),'BqqT15zsnzFZhyBJ3KMaEw==' (1)
All distinct Values,58 unique values,54453 unique values
Token Number of Distinct Values (Ignoring Missing Values),99 (0.00 %),80837 (3.48 %)
Token Most Common Value,'' (1282746),'' (1303891)
Token Least Common Value,'tg6g' (1),'BqqT15zsnzFZhyBJ3KMaEw' (1)
Token All distinct Values,99 unique values,80837 unique values
Text Length Mean,24.0,24.0


In [8]:
# [ ] TODO fcount bei Anzahlen einbauen
# [x] Profiling () -> Multiprocessing sinvoll?
# -> PyCharm Probleme... cProfile? PyCahrm lokal nicht wsl
# -> Profiling in python visualisierung in PyCahrm?
# [x] assertion, alle columns beschrieben? assert
# [ ] Texte verbessern
# [ ] String datentypen -> Tokens ("\W" oder ",") führen zu mehreren tokens, Rest Nominal
# [ ] TODO Neuen Gruppen: Zahlendaten, Nominale, Kategoriale, Wiederholte Nominale, Wiederholte Kategoriale, (Ordinal), nur den check code
# -> display für jeden?